# Causal Transformer: predicting shots and goals from event history

This experiment compares a Seq2Event-inspired Transformer with the constant,
logistic, XGBoost and two LSTM baselines. Each event mixes a categorical action with four continuous
features. Causal attention uses only the events observed so far in that possession.

Two independent models tune on **2025**, refit on **2023–2025** and evaluate on the
existing **2026 benchmark**. This notebook reads completed artifacts without fitting,
tuning or rescoring test data. Saved outputs are included. Run cells in order with
the project kernel. See [the full specification](../docs/transformer-baseline.md).

The architecture is inspired by [Seq2Event](https://eprints.soton.ac.uk/458099/);
the binary possession-outcome task differs from its original next-event prediction.

## 1. Experiment verification

The loader resolves repository paths, reads the predeclared search protocol
and verifies artifact hashes.
Completion metadata is present only after both final models and test comparisons
were saved successfully. The source hashes also identify the implementation used.

In [1]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p/'data/sources.json').is_file() and (p/'pyproject.toml').is_file()), None)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
experiment = project_root/'artifacts/transformer-2026'
if not (experiment/'summary.json').is_file():
    raise RuntimeError("Complete python -m afl_opposition.transformer_experiment first; use --resume if interrupted.")
summary = json.loads((experiment/'summary.json').read_text(encoding='utf-8'))
protocol = json.loads((experiment/'protocol.json').read_text(encoding='utf-8'))
for relative, expected in summary['artifact_sha256'].items():
    with (experiment/relative).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, relative
targets = protocol['targets']
refits = {t: json.loads((experiment/t/'history/refit.json').read_text(encoding='utf-8')) for t in targets}
print('Experiment:', experiment.relative_to(project_root).as_posix())
print('Verified artifacts:', len(summary['artifact_sha256']))
print('PyTorch:', protocol['torch_version'], 'Device:', summary['device'])
print('Both models frozen before test loading:', summary['both_models_frozen_before_test_loading'])

Experiment: artifacts/transformer-2026
Verified artifacts: 122
PyTorch: 2.11.0+cu128 Device: cuda
Both models frozen before test loading: True


## 2. Selected architectures

The input representation concatenates an action embedding with four scaled
continuous features projected through a Linear + ReLU layer. A second projection
maps the combined representation to `d_model`, followed by sinusoidal positions. Each causal Transformer block has multi-head attention and a feed-forward
hidden width of `d_model * ff_multiplier`. Final LayerNorm, dropout and a linear head
produce a logit at each real event; sigmoid gives a probability.

Pre-normalisation and GELU are implementation choices for this adaptation. The
table reports the independently selected shot and goal configurations. Dropout is
used on inputs, inside Transformer blocks and before the output head.

In [2]:
selected = []
for target, record in refits.items():
    selected.append(dict(target=target, **record['config'], epochs=record['epochs'],
        parameters=record['selected']['parameters'], validation_bce=record['selected']['validation_bce']))
display(pd.DataFrame(selected).set_index('target').T.rename_axis('setting').style.format('{:.6g}'))

target,shot,goal
setting,,
num_layers,3,3
d_model,32,128
num_heads,4,4
ff_multiplier,4,2
embedding_dim,4,4
continuous_dim,16,16
dropout,0.1,0
learning_rate,0.003,0.001
lambda_l2,1e-06,1e-06


## 3. Example input sequence and causal mask

The example contains a four-event development possession. Action IDs use the
goal model's training-only vocabulary; padding is ID 0 and unseen actions are ID 1. Coordinates and times remain
continuous. Numeric scaling is fitted only on the fitting partition.

In the attention table, each row is a query event and each column a possible key.
`True` means that event is visible. Event 2 can use events 0, 1 and 2, never event 3.
This is the **allowed** mask; the PyTorch API takes its inverse (True means blocked).
Positional encoding describes event order; the time features separately describe
elapsed time. The outcome and sample weight below illustrate labels, not features.

In [3]:
development = pd.read_parquet(project_root/'artifacts/goal-labels-2023-2025/examples.parquet')
sizes = development.groupby('possession_id').size()
chosen = sizes.loc[sizes.eq(4)].index[0]
example = development.loc[development.possession_id.eq(chosen)].sort_values('event_index').copy()
encoder = json.loads((experiment/'goal/history/selection/model.json').read_text(encoding='utf-8'))['encoder']
vocabulary = {name:i+2 for i, name in enumerate(encoder['categories'])}
example['action_id'] = example.action_type.map(vocabulary).fillna(1).astype(int)
display(example[['event_index','action_type','action_id','x_norm','y_norm','gap_seconds','elapsed_seconds','target','sample_weight']])
display(pd.DataFrame([[key <= query for key in range(4)] for query in range(4)],
    index=[f'Query {i}' for i in range(4)], columns=[f'Key {i}' for i in range(4)]))
display(sizes.describe(percentiles=[.5,.9,.99]).rename('eligible sequence length').to_frame())
del development

,event_index,action_type,action_id,x_norm,y_norm,gap_seconds,elapsed_seconds,target,sample_weight
30,0,gather,4,0.7375,-0.070922,0,0,0,0.25
31,1,handball,6,0.7,-0.042553,1,1,0,0.25
32,2,receive,13,0.5625,0.056738,1,2,0,0.25
33,3,kick,8,0.5875,0.113475,1,3,0,0.25


,Key 0,Key 1,Key 2,Key 3
Query 0,True,False,False,False
Query 1,True,True,False,False
Query 2,True,True,True,False
Query 3,True,True,True,True


,eligible sequence length
count,153822.000000
mean,5.491900
std,4.208851
min,1.000000
50%,4.000000
90%,11.000000
99%,20.000000
max,47.000000


## 4. Training objective and tuning protocol

$$J=\frac{1}{N}\sum_i\frac{1}{n_i}\sum_t\mathrm{BCE}(y_i,p_{i,t})
+\lambda\|\theta_{\mathrm{matrix}}\|_2^2.$$

Every possession has total loss weight one; padding has zero weight. Matrix weights
receive explicit L2; biases and LayerNorm scale vectors do not. Adam has zero weight
decay. The validation criterion is weighted BCE without regularisation or dropout.

Each target receives the same 24 candidates. All fit for at most 6 epochs; the best
6 continue to 18, and the best 2 to 48. Six epochs without improvement triggers early
stopping. These are cumulative budgets, not additional epochs. This bounded search
can prune slower-learning candidates. The batch size is 512 possessions.

In [4]:
display(pd.DataFrame(protocol['candidates']).rename_axis('trial'))
display(pd.DataFrame(protocol['stages'], columns=['cumulative_epoch_budget','candidates_retained']))
print('Batch size:', protocol['batch_size'], 'Training seed:', protocol['seed'])
print('Early-stopping patience:', protocol['patience'])
print('Optimiser:', protocol['optimiser'])
print('Attention backend:', protocol['attention_backend'])

,num_layers,d_model,num_heads,ff_multiplier,embedding_dim,continuous_dim,dropout,learning_rate,lambda_l2
trial,,,,,,,,,
0,1,64,4,2,8,8,0.10,0.0010,0.000010
1,3,128,4,2,4,16,0.00,0.0010,0.000001
2,3,64,4,2,8,8,0.25,0.0030,0.000001
3,2,32,8,4,16,16,0.25,0.0010,0.000000
4,1,64,8,2,4,8,0.10,0.0003,0.000000
5,2,64,8,2,8,16,0.25,0.0030,0.000100
6,2,128,2,4,16,8,0.00,0.0030,0.000100
7,2,64,4,2,8,16,0.00,0.0003,0.000000
8,3,128,8,4,16,8,0.25,0.0003,0.000010


,cumulative_epoch_budget,candidates_retained
0,6,6
1,18,2
2,48,1


Batch size: 512 Training seed: 2026
Early-stopping patience: 6
Optimiser: Adam with zero weight_decay; explicit L2
Attention backend: math


## 5. Candidate validation results

All candidates fit 2023–2024, including their vocabulary and numeric scaler. Selection
uses the best observed 2025 BCE; exact ties favour fewer parameters, then the earlier
trial. `stage` and `attempted_epochs` expose unequal budgets after pruning. The five
best candidates for each target are shown; complete tables remain in the artifacts.

In [5]:
for target in targets:
    search = pd.read_parquet(experiment/target/'history/search.parquet')
    assert search.selected.sum() == 1
    assert search.loc[search.selected,'validation_bce'].iloc[0] == search.validation_bce.min()
    print(target)
    display(search.sort_values('validation_bce').head(5)[['trial','num_layers','d_model','num_heads',
        'ff_multiplier','embedding_dim','continuous_dim','dropout','learning_rate','lambda_l2',
        'stage','attempted_epochs','best_epoch','validation_bce','selected']]
        .style.format({'validation_bce':'{:.6f}','lambda_l2':'{:.2g}'}))

shot


,trial,num_layers,d_model,num_heads,ff_multiplier,embedding_dim,continuous_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
18,18,3,32,4,4,4,16,0.100000,0.003000,1e-06,3,22,16,0.390703,True
20,20,1,64,4,4,8,16,0.100000,0.003000,1e-05,3,22,16,0.390924,False
2,2,3,64,4,2,8,8,0.250000,0.003000,1e-06,2,18,15,0.391702,False
12,12,1,64,2,2,8,16,0.100000,0.003000,1e-05,2,15,9,0.392290,False
10,10,3,128,2,2,16,16,0.000000,0.001000,1e-06,2,14,8,0.392532,False


goal


,trial,num_layers,d_model,num_heads,ff_multiplier,embedding_dim,continuous_dim,dropout,learning_rate,lambda_l2,stage,attempted_epochs,best_epoch,validation_bce,selected
1,1,3,128,4,2,4,16,0.000000,0.001000,1e-06,3,15,9,0.278519,True
13,13,2,128,8,4,16,8,0.100000,0.001000,1e-06,3,22,16,0.278666,False
10,10,3,128,2,2,16,16,0.000000,0.001000,1e-06,2,15,9,0.278908,False
20,20,1,64,4,4,8,16,0.100000,0.003000,1e-05,2,15,9,0.279374,False
12,12,1,64,2,2,8,16,0.100000,0.003000,1e-05,2,15,9,0.279393,False


## 6. Selected-model learning curves

The table includes the first, every fourth, best and final attempted epochs. Training BCE is an
online minibatch average with dropout active and weights changing. Validation BCE
is measured after the epoch with fixed weights and dropout disabled. These are
different measurement conditions, not directly comparable estimates of overfitting.

The minimum validation epoch sets refit duration. `best_at_budget` indicates that
the selected checkpoint is at the declared stage budget rather than an interior minimum.

In [6]:
for target, record in refits.items():
    winner = record['selected']
    curve = pd.read_parquet(experiment/target/'history/learning_curves.parquet')
    curve = curve.loc[curve.trial.eq(winner['trial'])]
    mask = curve.epoch.isin([1,winner['best_epoch'],winner['attempted_epochs']]) | curve.epoch.mod(4).eq(0)
    shown = curve.loc[mask,['epoch','training_bce','validation_bce','seconds']].copy()
    shown['selected_epoch'] = shown.epoch.eq(winner['best_epoch'])
    print(target, '— best at budget:', winner['best_at_budget'])
    display(shown.style.format({'training_bce':'{:.6f}','validation_bce':'{:.6f}','seconds':'{:.2f}'}))

shot — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
146,1,0.466196,0.424615,10.39,False
149,4,0.393047,0.414411,9.79,False
153,8,0.389375,0.395221,10.50,False
157,12,0.390268,0.408466,10.83,False
161,16,0.386715,0.390703,10.60,True
165,20,0.386892,0.393299,11.51,False
167,22,0.387116,0.390986,10.50,False


goal — best at budget: False


,epoch,training_bce,validation_bce,seconds,selected_epoch
6,1,0.312760,0.291166,10.34,False
9,4,0.277869,0.299810,9.84,False
13,8,0.274652,0.279185,10.18,False
14,9,0.273429,0.278519,10.24,True
17,12,0.273457,0.291790,10.30,False
20,15,0.272940,0.278950,10.36,False


## 7. Refitting and benchmark coverage

Each refit starts with new weights and a new encoder fitted on 2023–2025.
Training uses the selected epoch count and L2 coefficient, without validation
stopping.
Both models are saved before the first 2026 load and remain frozen during scoring.

The test event IDs, labels, weights and constant probabilities match earlier baselines.
The table counts event states, which are correlated within possessions and matches.
Each possession's event weights sum to one.

In [7]:
assert summary['both_models_frozen_before_test_loading']
assert summary['models_unchanged_during_evaluation']
coverage = []
for target, record in refits.items():
    predictions = pd.read_parquet(experiment/target/'history/test/predictions.parquet')
    assert predictions.groupby('possession_id').sample_weight.sum().sub(1).abs().max() < 1e-12
    assert record['epochs'] == record['selected']['best_epoch']
    assert record['config']['lambda_l2'] == record['selected']['lambda_l2']
    coverage.append(dict(target=target, refit_possessions=record['refit_possessions'],
        test_matches=predictions.match_id.nunique(), test_possessions=predictions.possession_id.nunique(),
        test_prefixes=len(predictions),
        test_positive_rate=(predictions.target*predictions.sample_weight).sum()/predictions.sample_weight.sum()))
display(pd.DataFrame(coverage))

,target,refit_possessions,test_matches,test_possessions,test_prefixes,test_positive_rate
0,shot,153822,218,51234,290909,0.216341
1,goal,153822,218,51234,290909,0.105809


## 8. Six-model comparison on the 2026 benchmark

Lower BCE and Brier are better; higher average precision is better. All metrics in
this table use possession weights. Classification metrics use threshold 0.5.
Comparisons are within a target because shot and goal prevalence differ. The constant predictor's
AP equals that target's positive prevalence, not 0.5.

In [8]:
comparisons = {t:pd.read_parquet(experiment/t/'comparison/metrics.parquet') for t in targets}
metrics = pd.concat([frame.assign(target=t) for t,frame in comparisons.items()], ignore_index=True)
numeric_metrics = ['log_loss','brier_score','average_precision','accuracy','precision','recall',
                   'specificity','f1','balanced_accuracy']
display(metrics[['target','model',*numeric_metrics]]
    .style.format({c:'{:.6f}' for c in numeric_metrics}, na_rep='undefined'))

,target,model,log_loss,brier_score,average_precision,accuracy,precision,recall,specificity,f1,balanced_accuracy
0,shot,constant,0.523087,0.169815,0.216341,0.783659,undefined,0.000000,1.000000,0.000000,0.500000
1,shot,logistic,0.423148,0.134152,0.555929,0.815618,0.672837,0.287535,0.961403,0.402895,0.624469
2,shot,xgboost,0.398320,0.125400,0.612536,0.829024,0.756987,0.308838,0.972630,0.438695,0.640734
3,shot,lstm_no_history,0.401857,0.126606,0.606363,0.827301,0.724644,0.325362,0.965869,0.449087,0.645616
4,shot,lstm_history,0.394988,0.124298,0.618982,0.829832,0.716298,0.353392,0.961360,0.473285,0.657376
5,shot,seq2event_transformer,0.417787,0.133067,0.610934,0.814198,0.586665,0.477774,0.907072,0.526650,0.692423
6,goal,constant,0.337951,0.094666,0.105809,0.894191,undefined,0.000000,1.000000,0.000000,0.500000
7,goal,logistic,0.297569,0.085786,0.284618,0.894596,0.589756,0.012575,0.998965,0.024625,0.505770
8,goal,xgboost,0.287580,0.082488,0.342662,0.899497,0.657114,0.104869,0.993525,0.180872,0.549197
9,goal,lstm_no_history,0.288036,0.082635,0.340169,0.899156,0.657531,0.097918,0.993965,0.170452,0.545941


## 9. Raw and weighted confusion matrices

Rows are actual classes and columns predicted classes: `[[TN, FP], [FN, TP]]`.
Raw entries count event states. Weighted entries sum prefix weights, so the total
equals the number of possessions although a prediction is made at every event.
The decision threshold is fixed at 0.5; it was not tuned on the test data.

In [9]:
for target in targets:
    confusion = pd.read_parquet(experiment/target/'history/test/confusion.parquet')
    for weighting in ['unweighted','possession_weighted']:
        row = confusion.loc[confusion.model.eq('seq2event_transformer') & confusion.weighting.eq(weighting)].iloc[0]
        matrix = pd.DataFrame([[row.tn,row.fp],[row.fn,row.tp]],
            index=[f'Actual no {target}',f'Actual {target}'], columns=[f'Predicted no {target}',f'Predicted {target}'])
        print(target, weighting, '— threshold 0.5')
        display(matrix.style.format('{:.0f}' if weighting == 'unweighted' else '{:.2f}'))

shot unweighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,185345,23450
Actual shot,51738,30376


shot possession_weighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,36418.95,3731.05
Actual shot,5788.35,5295.65


goal unweighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,246718,2807
Actual goal,37683,3701


goal possession_weighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,45295.17,517.83
Actual goal,4691.75,729.25


## 10. Differences from the strongest earlier models

Each difference is the Transformer metric minus the earlier model's metric. Negative BCE/Brier
and positive AP differences favour the Transformer. This is a model-family comparison:
the Transformer also changes continuous projections, positional representation and
batch size. It does not isolate attention alone. Independently tuned architectures
and one training seed limit causal or statistical claims about small differences.

In [10]:
differences = []
for target, frame in comparisons.items():
    indexed = frame.set_index('model')
    for baseline in ['lstm_history','xgboost']:
        differences.append(dict(target=target, compared_with=baseline,
            **{metric:indexed.loc['seq2event_transformer',metric]-indexed.loc[baseline,metric]
               for metric in ['log_loss','brier_score','average_precision']}))
display(pd.DataFrame(differences).style.format({m:'{:+.6f}' for m in ['log_loss','brier_score','average_precision']}))

,target,compared_with,log_loss,brier_score,average_precision
0,shot,lstm_history,+0.022799,+0.008769,-0.008049
1,shot,xgboost,+0.019467,+0.007667,-0.001602
2,goal,lstm_history,+0.003686,+0.000932,-0.006673
3,goal,xgboost,+0.001801,+0.000546,-0.002103


## 11. Calibration and interpretation limits

Calibration bands compare mean predicted probability with the observed outcome rate.
All averages use possession weights. Sparse high-probability bands deserve cautious
interpretation. These tables are diagnostics; no calibration model or threshold is
fitted here.

The 2026 season has already been observed for previous models and remains a shared
benchmark. These point estimates do not establish significance, live match utility
or that Transformers are generally better than LSTMs. Labels and possession boundaries
remain inferred from the public data. Independent shot/goal models do not enforce
probability ordering. Further temporal validation would strengthen deployment claims.

The shot Transformer's weighted mean test probability is **28.4%**, compared with an observed shot rate of **21.6%**. In the 0.5–0.6 probability band, the mean prediction is 54.7% while the observed shot rate is 39.0%. The calibration table shows overprediction, alongside the poorer shot BCE and Brier score. Higher recall at threshold 0.5 therefore does not imply better probability estimates. These test observations are reported without recalibrating or retuning either model.

In [11]:
for target in targets:
    calibration = pd.read_parquet(experiment/target/'history/test/calibration.parquet')
    calibration = calibration.loc[calibration.model.eq('seq2event_transformer')].copy()
    observed = f'observed_{target}_rate'
    calibration['prediction_minus_observed'] = calibration.mean_prediction-calibration[observed]
    print(target)
    display(calibration[['bin_lower','bin_upper','examples','weight_sum','mean_prediction',observed,'prediction_minus_observed']]
        .style.format({'weight_sum':'{:.1f}','mean_prediction':'{:.4f}',observed:'{:.4f}','prediction_minus_observed':'{:+.4f}'}))

shot


,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_shot_rate,prediction_minus_observed
1,0.000000,0.100000,66970,14837.8,0.0557,0.0413,+0.0144
2,0.100000,0.200000,55917,9375.7,0.1481,0.1091,+0.0390
3,0.200000,0.300000,50396,8254.1,0.2463,0.1781,+0.0682
4,0.300000,0.400000,37638,5710.9,0.3482,0.2484,+0.0998
5,0.400000,0.500000,26162,4028.8,0.4473,0.3136,+0.1337
6,0.500000,0.600000,19846,2879.3,0.5470,0.3896,+0.1574
7,0.600000,0.700000,13302,1897.3,0.6458,0.4826,+0.1632
8,0.700000,0.800000,5753,871.5,0.7461,0.5698,+0.1763
9,0.800000,0.900000,6977,1659.3,0.8576,0.6825,+0.1751
10,0.900000,1.000000,7948,1719.3,0.9619,0.9476,+0.0143


goal


,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_goal_rate,prediction_minus_observed
1,0.000000,0.100000,169872,32921.7,0.0387,0.0480,-0.0093
2,0.100000,0.200000,67422,10312.2,0.1419,0.1401,+0.0018
3,0.200000,0.300000,29020,4019.7,0.2438,0.2059,+0.0379
4,0.300000,0.400000,12887,1877.5,0.3431,0.2716,+0.0715
5,0.400000,0.500000,5200,855.9,0.4450,0.3850,+0.0601
6,0.500000,0.600000,2996,548.0,0.5483,0.4591,+0.0893
7,0.600000,0.700000,2336,477.9,0.6482,0.6311,+0.0171
8,0.700000,0.800000,1176,221.3,0.7269,0.7960,-0.0691
